Cross-Matching Multiple Source Catalogs in jdaviz

Treat the first catalog as the base. Match each subsequent catalog against the running merged set. This generalizes to N catalogs and keeps every step a well-understood pairwise problem.

Two matching modes per pair:
- id — exact join on a shared source-id column (fast, unambiguous when ids are global).
- sky — SkyCoord.match_to_catalog_sky + a separation tolerance.
- id_then_sky — use ids where present, fall back to sky for the rest.

Ambiguity detection (the part that needs user confirmation):
- a nearest neighbor whose separation is in a "grey zone" (tolerance < sep <= review_factor*tolerance),
- many-to-one collisions: two base sources whose best match is the same other-catalog source.
These are returned as a separate review table rather than silently accepted/rejected.

In [1]:
import uuid

import numpy as np
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import QTable

rng = np.random.default_rng(42)


def _get_skycoord(table):
    """Best-effort SkyCoord extraction, mirroring Catalogs._file_parser conventions."""
    if 'sky_centroid' in table.colnames:
        return SkyCoord(table['sky_centroid'])
    ra_candidates = ['Right Ascension (degrees)', 'ra', 'RA']
    dec_candidates = ['Declination (degrees)', 'dec', 'DEC']
    ra_col = next((c for c in ra_candidates if c in table.colnames), None)
    dec_col = next((c for c in dec_candidates if c in table.colnames), None)
    if ra_col is None or dec_col is None:
        raise ValueError('Catalog needs sky_centroid or RA/Dec columns.')
    # u.Quantity respects an existing unit and assumes deg only when unitless
    ra = u.Quantity(table[ra_col], u.deg)
    dec = u.Quantity(table[dec_col], u.deg)
    return SkyCoord(ra=ra, dec=dec)


def _get_object_id(table, name, id_columns, row_idx):
    """Return the source id for ``row_idx`` of ``table``.

    Uses the catalog's id column (from ``id_columns``) when available, otherwise
    falls back to a generated UUID so every source still has a unique handle.
    """
    id_columns = id_columns or {}
    col = id_columns.get(name)
    if col is not None and col in table.colnames:
        return str(np.asarray(table[col])[row_idx])
    return str(uuid.uuid4())


In [2]:
def crossmatch_pair(base_coords, other_coords, tolerance=1 * u.arcsec,
                    review_factor=2.0):
    """Positionally match `other` onto `base`.

    Returns a dict with three index arrays describing, for each base source:
      - match_idx : index into `other` of the nearest neighbor (or -1)
      - sep       : separation to that neighbor (arcsec, NaN if none)
      - status    : 'matched' | 'review' | 'unmatched'
    plus `collisions`: set of base indices that share a best-match `other` source.
    """
    n_base = len(base_coords)
    match_idx = np.full(n_base, -1, dtype=int)
    sep_arcsec = np.full(n_base, np.nan)
    status = np.array(['unmatched'] * n_base, dtype=object)

    if len(other_coords) == 0 or n_base == 0:
        return dict(match_idx=match_idx, sep=sep_arcsec, status=status, collisions=set())

    idx, sep2d, _ = base_coords.match_to_catalog_sky(other_coords)
    within = sep2d <= tolerance
    review = (sep2d > tolerance) & (sep2d <= review_factor * tolerance)

    match_idx[within | review] = idx[within | review]
    sep_arcsec[within | review] = sep2d[within | review].to_value(u.arcsec)
    status[within] = 'matched'
    status[review] = 'review'

    # many-to-one collisions: same `other` index claimed by multiple base sources
    claimed = match_idx[match_idx >= 0]
    _, counts = np.unique(claimed, return_counts=True)
    dup_targets = set(np.unique(claimed)[counts > 1].tolist())
    collisions = {i for i in range(n_base)
                  if match_idx[i] in dup_targets}
    for i in collisions:
        status[i] = 'review'

    return dict(match_idx=match_idx, sep=sep_arcsec, status=status, collisions=collisions)

In [3]:
def crossmatch_catalogs(catalogs, tolerance=1 * u.arcsec, review_factor=2.0,
                        id_columns=None, mode='id_then_sky', join='outer'):
    """Cross-match N catalogs.

    Parameters
    ----------
    catalogs : list of (name, QTable)
        Ordered; first is the base. Each table must yield a SkyCoord.
    tolerance : Quantity (angle)
        Hard positional match radius.
    review_factor : float
        Matches with tolerance < sep <= review_factor*tolerance are flagged for review.
    id_columns : dict or None
        {catalog_name: id_column_name} for exact id matching.
    mode : {'id', 'sky', 'id_then_sky'}
    join : {'outer', 'left'}
        'outer' (default) additionally appends those unmatched sources as new rows with
        ``base_idx == -1`` (a full N-way union). Note: this simple mockup does
        not merge unmatched sources *across* the non-base catalogs with each
        other; each becomes its own row.
        'left' keeps exactly one row per base source; sources in
        catalogs 2..N that match no base source are dropped.


    Returns
    -------
    merged : QTable
        One row per base source (plus appended rows if ``join='outer'``), with
        an ``object_id`` column, <name>_idx and <name>_sep_arcsec columns, a
        match_count, and a needs_review flag. ``object_id`` is taken from the
        originating catalog's id column when available, otherwise a generated
        UUID. For appended (outer-join) rows the id comes from the catalog the
        source was added from.
    review : QTable
        Subset of rows (with context) that need user confirmation.
    """
    if join not in ('left', 'outer'):
        raise ValueError("join must be 'left' or 'outer'")
    id_columns = id_columns or {}
    base_name, base_tbl = catalogs[0]
    base_coords = _get_skycoord(base_tbl)
    n_base = len(base_tbl)
    other_names = [name for name, _ in catalogs[1:]]

    merged = QTable()
    merged['base_idx'] = np.arange(n_base)
    # object-dtype so appended UUIDs (36 chars) aren't truncated to the base id width
    merged['object_id'] = np.array(
        [_get_object_id(base_tbl, base_name, id_columns, i) for i in range(n_base)],
        dtype=object)
    merged['base_ra'] = base_coords.ra.deg * u.deg
    merged['base_dec'] = base_coords.dec.deg * u.deg
    merged[f'{base_name}_idx'] = np.arange(n_base)
    needs_review = np.zeros(n_base, dtype=bool)
    match_count = np.ones(n_base, dtype=int)

    # remember, per catalog, which `other` rows were consumed (for outer join)
    used_by_cat = {}
    coords_by_cat = {}

    for name, tbl in catalogs[1:]:
        other_coords = _get_skycoord(tbl)
        coords_by_cat[name] = other_coords
        match_idx = np.full(n_base, -1, dtype=int)
        sep = np.full(n_base, np.nan)
        status = np.array(['unmatched'] * n_base, dtype=object)

        # 1) exact id matching where requested and possible
        used_other = set()
        if mode in ('id', 'id_then_sky') and base_name in id_columns and name in id_columns:
            base_ids = np.asarray(base_tbl[id_columns[base_name]]).astype(str)
            other_ids = np.asarray(tbl[id_columns[name]]).astype(str)
            lut = {v: j for j, v in enumerate(other_ids)}
            for i, bid in enumerate(base_ids):
                j = lut.get(bid)
                if j is not None:
                    match_idx[i] = j
                    sep[i] = base_coords[i].separation(other_coords[j]).arcsec
                    status[i] = 'matched'
                    used_other.add(j)

        # 2) positional fallback for the still-unmatched base rows
        if mode in ('sky', 'id_then_sky'):
            todo = np.where(match_idx < 0)[0]
            if len(todo):
                avail = [j for j in range(len(tbl)) if j not in used_other]
                if avail:
                    res = crossmatch_pair(base_coords[todo], other_coords[avail],
                                          tolerance=tolerance, review_factor=review_factor)
                    for k, i in enumerate(todo):
                        if res['match_idx'][k] >= 0:
                            j = avail[res['match_idx'][k]]
                            match_idx[i] = j
                            sep[i] = res['sep'][k]
                            status[i] = res['status'][k]
                            used_other.add(j)

        merged[f'{name}_idx'] = match_idx
        merged[f'{name}_sep_arcsec'] = sep
        merged[f'{name}_status'] = status
        needs_review |= (status == 'review')
        match_count += (match_idx >= 0).astype(int)
        used_by_cat[name] = used_other

    merged['match_count'] = match_count
    merged['needs_review'] = needs_review

    # outer join: append other-catalog sources that matched no base source
    if join == 'outer':
        # `<base_name>_idx` columns stay -1 (absent from the base catalog), but
        # `base_idx` keeps counting so every appended row gets a unique id.
        idx_cols = [f'{base_name}_idx'] + [f'{n}_idx' for n in other_names]
        next_base_idx = n_base
        for name, tbl in catalogs[1:]:
            oc = coords_by_cat[name]
            unmatched = [j for j in range(len(tbl)) if j not in used_by_cat[name]]
            for j in unmatched:
                row = {col: -1 for col in idx_cols}
                row['base_idx'] = next_base_idx
                next_base_idx += 1
                # id comes from the catalog the source was added from (or a UUID)
                row['object_id'] = _get_object_id(tbl, name, id_columns, j)
                row['base_ra'] = oc[j].ra.deg * u.deg
                row['base_dec'] = oc[j].dec.deg * u.deg
                for n in other_names:
                    row[f'{n}_sep_arcsec'] = np.nan
                    # 'absent' = column n was not involved for this appended row
                    row[f'{n}_status'] = 'matched' if n == name else 'absent'
                row[f'{name}_idx'] = j
                row['match_count'] = 1
                row['needs_review'] = False
                merged.add_row(row)

    review = merged[merged['needs_review']]
    return merged, review

1.1 Synthetic catalogs to exercise the engine
We build three catalogs around the same field:

A (base): 8 sources, with a global source_id.
B: same field, shares ids for some sources, positions jittered by ~0.3″, plus an extra source.
C: position-only (no shared ids), with one deliberately ambiguous pair to trigger review.

In [4]:
base_ra = 150.0 + rng.uniform(-0.01, 0.01, size=8)
base_dec = 2.0 + rng.uniform(-0.01, 0.01, size=8)
cat_a = QTable({
    'source_id': [f'A{i:03d}' for i in range(8)],
    'ra': base_ra * u.deg,
    'dec': base_dec * u.deg,
})

# B: jitter ~0.3 arcsec (~8.3e-5 deg), share ids for first 5 sources
jit = 0.3 / 3600.0
cat_b = QTable({
    'source_id': [f'A{i:03d}' for i in range(5)] + ['B900', 'B901', 'B902'],
    'ra': np.concatenate([base_ra[:5] + rng.uniform(-jit, jit, 5),
                          150.0 + rng.uniform(-0.01, 0.01, 3)]) * u.deg,
    'dec': np.concatenate([base_dec[:5] + rng.uniform(-jit, jit, 5),
                           2.0 + rng.uniform(-0.01, 0.01, 3)]) * u.deg,
})

# C: position only. Source 6 is placed in the grey zone (~1.5 arcsec) to force review.
grey = 1.5 / 3600.0
c_ra = base_ra.copy()
c_dec = base_dec.copy()
c_ra[6] += grey
cat_c = QTable({
    'Right Ascension (degrees)': c_ra * u.deg,
    'Declination (degrees)': c_dec * u.deg,
})

catalogs = [('A', cat_a), ('B', cat_b), ('C', cat_c)]
len(cat_a), len(cat_b), len(cat_c)

(8, 8, 8)

In [5]:
catalogs

[('A',
  <QTable length=8>
  source_id         ra                dec        
                   deg                deg        
     str4        float64            float64      
  --------- ------------------ ------------------
       A000 150.00547912097113 1.9925622726535108
       A001 149.99877756879505 1.9990077187579114
       A002 150.00717195839823 1.9974159604846515
       A003 150.00394736058118  2.008535299776972
       A004 149.99188354695775 2.0028773024016133
       A005 150.00951244703273 2.0064552322654166
       A006  150.0052227940398 1.9988682839765466
       A007 150.00572128610554 1.9945447744356954),
 ('B',
  <QTable length=8>
  source_id         ra                dec        
                   deg                deg        
     str4        float64            float64      
  --------- ------------------ ------------------
       A000 150.00548821843563 1.9926086699030232
       A001 149.99870487167107   1.99895682520922
       A002 150.00722656359358 1.99741041398

In [6]:
merged, review = crossmatch_catalogs(
    catalogs,
    tolerance=1 * u.arcsec,
    review_factor=2.0,
    id_columns={'A': 'source_id', 'B': 'source_id'},
    mode='id_then_sky',
)
merged

base_idx,object_id,base_ra,base_dec,A_idx,B_idx,B_sep_arcsec,B_status,C_idx,C_sep_arcsec,C_status,match_count,needs_review
,,deg,deg,,,,,,,,,
int64,object,float64,float64,int64,int64,float64,object,int64,float64,object,int64,bool
0,A000,150.00547912097113,1.9925622726535108,0,0,0.17020686410955332,matched,0,0.0,matched,3,False
1,A001,149.99877756879505,1.9990077187579114,1,1,0.3193383614879235,matched,1,0.0,matched,3,False
2,A002,150.00717195839823,1.9974159604846515,2,2,0.1974713618111863,matched,2,0.0,matched,3,False
3,A003,150.00394736058118,2.008535299776972,3,3,0.28487632528196527,matched,3,0.0,matched,3,False
4,A004,149.99188354695775,2.0028773024016133,4,4,0.2587966843260676,matched,4,0.0,matched,3,False
5,A005,150.00951244703273,2.0064552322654166,5,-1,nan,unmatched,5,0.0,matched,2,False
6,A006,150.0052227940398,1.9988682839765466,6,-1,nan,unmatched,6,1.4990872742460732,review,2,True
7,A007,150.00572128610554,1.9945447744356954,7,-1,nan,unmatched,7,0.0,matched,2,False


In [7]:
print('Rows needing user confirmation:', len(review))
review

Rows needing user confirmation: 1


base_idx,object_id,base_ra,base_dec,A_idx,B_idx,B_sep_arcsec,B_status,C_idx,C_sep_arcsec,C_status,match_count,needs_review
,,deg,deg,,,,,,,,,
int64,object,float64,float64,int64,int64,float64,object,int64,float64,object,int64,bool
6,A006,150.0052227940398,1.9988682839765466,6,-1,nan,unmatched,6,1.4990872742460732,review,2,True


1.2 Applying a user decision
The review table is what gets surfaced in the UI. The user accepts/rejects each flagged candidate; the helper below applies those decisions back onto the merged table.

In [9]:
def apply_review_decisions(merged, decisions):
    """decisions: {base_idx: {catalog_name: bool_accept}} -> updated merged table."""
    out = merged.copy()
    for base_idx, per_cat in decisions.items():
        row = np.where(out['base_idx'] == base_idx)[0][0]
        for name, accept in per_cat.items():
            if not accept:
                out[f'{name}_idx'][row] = -1
                out[f'{name}_sep_arcsec'][row] = np.nan
                out[f'{name}_status'][row] = 'rejected'
    # recompute summary columns
    idx_cols = [c for c in out.colnames if c.endswith('_idx') and c != 'base_idx']
    out['match_count'] = np.sum([(out[c] >= 0).astype(int) for c in idx_cols], axis=0)
    status_cols = [c for c in out.colnames if c.endswith('_status')]
    out['needs_review'] = np.any([out[c] == 'review' for c in status_cols], axis=0)
    return out


# Example: user confirms C matches base source 6 after all.
decisions = {6: {'C': True}}
resolved = apply_review_decisions(merged, decisions)
resolved[['base_idx', 'A_idx', 'B_idx', 'C_idx', 'match_count', 'needs_review']]

base_idx,A_idx,B_idx,C_idx,match_count,needs_review
int64,int64,int64,int64,int64,bool
0,0,0,0,3,False
1,1,1,1,3,False
2,2,2,2,3,False
3,3,3,3,3,False
4,4,4,4,3,False
5,5,-1,5,2,False
6,6,-1,6,2,True
7,7,-1,7,2,False
8,-1,5,-1,1,False


A tray plugin under category data:analysis, sitting next to Catalog Search. It can match any set of already-loaded catalogs, re-run interactively, and own the review UI. Add a metadata-association mode that groups loaded data by a chosen metadata key (meta['OBJECT'], proposal id, etc.) and/or sky footprint overlap.

### Suggested ticket order
1. Create jdaviz/core/crossmatch.py by building on the previous code. Also include unit tests.
2. Add the Catalog Cross-Match plugin and connect it to the crossmatch code
3. Add the review/confirmation sub-panel
4. Add the ability to link spectrum/2D spectrum/cube to a row so that data is populated